In [3]:
using StaticArrays
using CUDA

rates = SVector{85,Float32}(0.00129, 0.00143, 0.00154, 0.00160, 0.00166, 0.00168, 0.00167, 0.00164, 0.00161, 0.00157, 0.00152, 0.00148,
    0.00146, 0.00144, 0.00144, 0.00144, 0.00147, 0.00150, 0.00155, 0.00161, 0.00169, 0.00177, 0.00188, 0.00200,
    0.00214, 0.00229, 0.00247, 0.00265, 0.00286, 0.00307, 0.00332, 0.00359, 0.00388, 0.00419, 0.00454, 0.00491,
    0.00535, 0.00586, 0.00643, 0.00709, 0.00782, 0.00863, 0.00949, 0.01042, 0.01147, 0.01264, 0.01394, 0.01542,
    0.01711, 0.01902, 0.02113, 0.02340, 0.02586, 0.02850, 0.03138, 0.03463, 0.03831, 0.04256, 0.04744, 0.05292,
    0.05880, 0.06506, 0.07164, 0.07847, 0.08572, 0.09367, 0.10252, 0.11252, 0.12379, 0.13611, 0.14920, 0.16280,
    0.17679, 0.19089, 0.20529, 0.22019, 0.23584, 0.25275, 0.27163, 0.29565, 0.32996, 0.38455, 0.48020, 0.65798,
    1.00000)
rates2 = SVector{85,Float32}(0.00129, 0.00143, 0.00154, 0.00160, 0.00166, 0.00168, 0.00167, 0.00164, 0.00161, 0.00157, 0.00152, 0.00148,
    0.00146, 0.00144, 0.00144, 0.00144, 0.00147, 0.00150, 0.00155, 0.00161, 0.00169, 0.00177, 0.00188, 0.00200,
    0.00214, 0.00229, 0.00247, 0.00265, 0.00286, 0.00307, 0.00332, 0.00359, 0.00388, 0.00419, 0.00454, 0.00491,
    0.00535, 0.00586, 0.00643, 0.00709, 0.00782, 0.00863, 0.00949, 0.01042, 0.01147, 0.01264, 0.01394, 0.01542,
    0.01711, 0.01902, 0.02113, 0.02340, 0.02586, 0.02850, 0.03138, 0.03463, 0.03831, 0.04256, 0.04744, 0.05292,
    0.05880, 0.06506, 0.07164, 0.07847, 0.08572, 0.09367, 0.10252, 0.11252, 0.12379, 0.13611, 0.14920, 0.16280,
    0.17679, 0.19089, 0.20529, 0.22019, 0.23584, 0.25275, 0.27163, 0.29565, 0.32996, 0.38455, 0.48020, 0.65798,
    1.00000)
rates3 = SVector{85,Float32}(0.00129, 0.00143, 0.00154, 0.00160, 0.00166, 0.00168, 0.00167, 0.00164, 0.00161, 0.00157, 0.00152, 0.00148,
    0.00146, 0.00144, 0.00144, 0.00144, 0.00147, 0.00150, 0.00155, 0.00161, 0.00169, 0.00177, 0.00188, 0.00200,
    0.00214, 0.00229, 0.00247, 0.00265, 0.00286, 0.00307, 0.00332, 0.00359, 0.00388, 0.00419, 0.00454, 0.00491,
    0.00535, 0.00586, 0.00643, 0.00709, 0.00782, 0.00863, 0.00949, 0.01042, 0.01147, 0.01264, 0.01394, 0.01542,
    0.01711, 0.01902, 0.02113, 0.02340, 0.02586, 0.02850, 0.03138, 0.03463, 0.03831, 0.04256, 0.04744, 0.05292,
    0.05880, 0.06506, 0.07164, 0.07847, 0.08572, 0.09367, 0.10252, 0.11252, 0.12379, 0.13611, 0.14920, 0.16280,
    0.17679, 0.19089, 0.20529, 0.22019, 0.23584, 0.25275, 0.27163, 0.29565, 0.32996, 0.38455, 0.48020, 0.65798,
    1.00000)

rates_gpu = CUDA.cu(rates)
rates2_gpu = CUDA.cu(rates2)
rates3_gpu = CUDA.cu(rates3)

@show typeof(rates_gpu)

@inline function example2(age, rates, rates2, rates3)
    return rates[age-14] + 0.01f0 * rates2[age-14] + 0.0025f0 * rates3[age-14]
end
function kernel_test2(age0, age1, rates, rates1, rates2, qres)

    sumqx = zero(Float32)
    for age = age0:age1
        sumqx += example2(age, rates, rates1, rates2)
    end
    qres[1] = sumqx

    nothing
end

qres = CUDA.zeros(Float32, 1)
cudakernel0 = @cuda launch = false kernel_test2(20, 90, rates_gpu, rates2_gpu, rates3_gpu, qres)
config = launch_configuration(cudakernel0.fun)
@show CUDA.registers(cudakernel0)
@show CUDA.memory(cudakernel0)

typeof(rates_gpu) = SVector{85, Float32}
CUDA.registers(cudakernel0) = 32
CUDA.memory(cudakernel0) = (local = 1056, shared = 0, constant = 0)


(local = 1056, shared = 0, constant = 0)

In [4]:
rates = Vector{Float32}([0.00129, 0.00143, 0.00154, 0.00160, 0.00166, 0.00168, 0.00167, 0.00164, 0.00161, 0.00157, 0.00152, 0.00148,
    0.00146, 0.00144, 0.00144, 0.00144, 0.00147, 0.00150, 0.00155, 0.00161, 0.00169, 0.00177, 0.00188, 0.00200,
    0.00214, 0.00229, 0.00247, 0.00265, 0.00286, 0.00307, 0.00332, 0.00359, 0.00388, 0.00419, 0.00454, 0.00491,
    0.00535, 0.00586, 0.00643, 0.00709, 0.00782, 0.00863, 0.00949, 0.01042, 0.01147, 0.01264, 0.01394, 0.01542,
    0.01711, 0.01902, 0.02113, 0.02340, 0.02586, 0.02850, 0.03138, 0.03463, 0.03831, 0.04256, 0.04744, 0.05292,
    0.05880, 0.06506, 0.07164, 0.07847, 0.08572, 0.09367, 0.10252, 0.11252, 0.12379, 0.13611, 0.14920, 0.16280,
    0.17679, 0.19089, 0.20529, 0.22019, 0.23584, 0.25275, 0.27163, 0.29565, 0.32996, 0.38455, 0.48020, 0.65798,
    1.00000])
rates2 = Vector{Float32}([0.00129, 0.00143, 0.00154, 0.00160, 0.00166, 0.00168, 0.00167, 0.00164, 0.00161, 0.00157, 0.00152, 0.00148,
    0.00146, 0.00144, 0.00144, 0.00144, 0.00147, 0.00150, 0.00155, 0.00161, 0.00169, 0.00177, 0.00188, 0.00200,
    0.00214, 0.00229, 0.00247, 0.00265, 0.00286, 0.00307, 0.00332, 0.00359, 0.00388, 0.00419, 0.00454, 0.00491,
    0.00535, 0.00586, 0.00643, 0.00709, 0.00782, 0.00863, 0.00949, 0.01042, 0.01147, 0.01264, 0.01394, 0.01542,
    0.01711, 0.01902, 0.02113, 0.02340, 0.02586, 0.02850, 0.03138, 0.03463, 0.03831, 0.04256, 0.04744, 0.05292,
    0.05880, 0.06506, 0.07164, 0.07847, 0.08572, 0.09367, 0.10252, 0.11252, 0.12379, 0.13611, 0.14920, 0.16280,
    0.17679, 0.19089, 0.20529, 0.22019, 0.23584, 0.25275, 0.27163, 0.29565, 0.32996, 0.38455, 0.48020, 0.65798,
    1.00000])
rates3 = Vector{Float32}([0.00129, 0.00143, 0.00154, 0.00160, 0.00166, 0.00168, 0.00167, 0.00164, 0.00161, 0.00157, 0.00152, 0.00148,
    0.00146, 0.00144, 0.00144, 0.00144, 0.00147, 0.00150, 0.00155, 0.00161, 0.00169, 0.00177, 0.00188, 0.00200,
    0.00214, 0.00229, 0.00247, 0.00265, 0.00286, 0.00307, 0.00332, 0.00359, 0.00388, 0.00419, 0.00454, 0.00491,
    0.00535, 0.00586, 0.00643, 0.00709, 0.00782, 0.00863, 0.00949, 0.01042, 0.01147, 0.01264, 0.01394, 0.01542,
    0.01711, 0.01902, 0.02113, 0.02340, 0.02586, 0.02850, 0.03138, 0.03463, 0.03831, 0.04256, 0.04744, 0.05292,
    0.05880, 0.06506, 0.07164, 0.07847, 0.08572, 0.09367, 0.10252, 0.11252, 0.12379, 0.13611, 0.14920, 0.16280,
    0.17679, 0.19089, 0.20529, 0.22019, 0.23584, 0.25275, 0.27163, 0.29565, 0.32996, 0.38455, 0.48020, 0.65798,
    1.00000])


rates_gpu = CUDA.cu(rates)
rates2_gpu = CUDA.cu(rates2)
rates3_gpu = CUDA.cu(rates3)

@show typeof(rates_gpu)

@inline function example2(age, rates, rates2, rates3)
    return rates[age-14] + 0.01f0 * rates2[age-14] + 0.0025f0 * rates3[age-14]
end
function kernel_test2(age0, age1, rates, rates1, rates2, qres)

    sumqx = zero(Float32)
    for age = age0:age1
        sumqx += example2(age, rates, rates1, rates2)
    end
    qres[1] = sumqx

    nothing
end

qres = CUDA.zeros(Float32, 1)
cudakernel0 = @cuda launch = false kernel_test2(20, 90, rates_gpu, rates2_gpu, rates3_gpu, qres)
config = launch_configuration(cudakernel0.fun)
@show CUDA.registers(cudakernel0)
@show CUDA.memory(cudakernel0)

typeof(rates_gpu) = CuArray{Float32, 1, CUDA.DeviceMemory}
CUDA.registers(cudakernel0) = 32
CUDA.memory(cudakernel0) = (local = 32, shared = 0, constant = 0)


(local = 32, shared = 0, constant = 0)